In [2]:
import pandas as pd
import re

# ===== 1. Load data =====
file_path = "/Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/ntsb_with_fulltext.csv"
df = pd.read_csv(file_path, encoding="utf-8", engine="python")

# ===== 2. Define target fields (order matters) =====
FIELDS = [
    "Location:",
    "Accident Number:",
    "Date & Time:",
    "Registration:",
    "Aircraft:",
    "Aircraft Damage:",
    "Defining Event:",
    "Injuries:",
    "Flight Conducted Under:",
    "Analysis",
    "Probable Cause and Findings",
    "Factual Information",
    "Pilot Information",
    "Aircraft and Owner/Operator Information",
    "Meteorological Information and Flight Plan",
    "Airport Information",
    "Wreckage and Impact Information",
    "Administrative Information",
]

# ===== 3. Precompile regex pattern =====
# We'll treat the text as sections separated by these headers.
pattern = r"|".join(
    [rf"(?P<{re.sub(r'[^A-Za-z0-9]+', '_', f)}>{re.escape(f)})" for f in FIELDS]
)

def extract_sections(text):
    """Extract each field's text block from the report."""
    if pd.isna(text):
        return {re.sub(r"[^A-Za-z0-9]+", "_", f): None for f in FIELDS}

    # Find all headers and their positions
    matches = [(m.lastgroup, m.start()) for m in re.finditer(pattern, text, flags=re.IGNORECASE)]
    sections = {}

    for i, (field, start) in enumerate(matches):
        end = matches[i + 1][1] if i + 1 < len(matches) else len(text)
        content = text[start:end].split(":", 1)[-1].strip()
        sections[field] = content

    # Fill missing fields with None
    for f in FIELDS:
        key = re.sub(r"[^A-Za-z0-9]+", "_", f)
        if key not in sections:
            sections[key] = None

    return sections

# ===== 4. Apply extraction & keep only in memory =====
records = df["fulltext"].apply(extract_sections).apply(pd.Series)

df_extracted_1 = pd.concat([df, records], axis=1)

print(f"Extraction complete. In-memory columns added: {len(df_extracted_1.columns)} total columns")

Extraction complete. In-memory columns added: 65 total columns


In [3]:
import pandas as pd
import re

# This cell expects `df_extracted_1` from cell 0 (no intermediate CSV is written).
FINAL_OUTPUT_FILE = "/Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/ntsb_with_extracted.csv"


def clean_date(date_str):
    """Removes T and Z from date strings like 2025-10-07T12:15:00Z"""
    if pd.isna(date_str):
        return date_str
    return str(date_str).replace("T", " ").replace("Z", "")


def extract_meteo_info(text):
    """Parse Meteorological Information to find key meteo fields."""
    if not isinstance(text, str):
        return {
            "Condition_of_Light": "Unknown",
            "Lowest_Cloud_Condition": "Unknown",
            "Wind_Info": "Unknown",
            "Lowest_Ceiling": "Unknown",
            "Visibility": "Unknown",
        }

    # 1. Condition of Light
    light_match = re.search(r"Condition of Light:\s*([A-Za-z\s]+)", text)
    light = light_match.group(1).strip() if light_match else "Unknown"

    # 2. Lowest Cloud Condition
    cloud_match = re.search(
        r"Lowest Cloud Condition:\s*(.*?)(?=\s+(?:Wind|Lowest|Visibility|Altimeter|Temperature|$))",
        text,
    )
    cloud = cloud_match.group(1).strip() if cloud_match else "Unknown"

    # 3. Wind Speed/Gusts, Direction
    wind_combined_match = re.search(
        r"Wind Speed/Gusts,\s*Direction:\s*(.*?)(?=\s+(?:Lowest|Visibility|Altimeter|$))",
        text,
    )

    if wind_combined_match:
        wind = wind_combined_match.group(1).strip()
    else:
        speed_match = re.search(
            r"Wind Speed/Gusts:\s*(.*?)(?=\s+(?:Wind Direction|Lowest|Visibility|$))",
            text,
        )
        dir_match = re.search(
            r"Wind Direction:\s*(.*?)(?=\s+(?:Lowest|Visibility|Altimeter|$))",
            text,
        )

        speed = speed_match.group(1).strip() if speed_match else ""
        direction = dir_match.group(1).strip() if dir_match else ""

        if speed or direction:
            wind = f"{speed}, {direction}".strip(", ")
        else:
            wind = "Unknown"

    # 4. Lowest Ceiling
    ceiling_match = re.search(
        r"Lowest Ceiling:\s*(.*?)(?=\s+(?:Visibility|Altimeter|Temperature|$))",
        text,
    )
    ceiling = ceiling_match.group(1).strip() if ceiling_match else "Unknown"

    # 5. Visibility
    vis_match = re.search(
        r"Visibility:\s*(.*?)(?=\s+(?:Altimeter|Temperature|Type of Flight|$))",
        text,
    )
    visibility = vis_match.group(1).strip() if vis_match else "Unknown"

    return {
        "Condition_of_Light": light,
        "Lowest_Cloud_Condition": cloud,
        "Wind_Info": wind,
        "Lowest_Ceiling": ceiling,
        "Visibility": visibility,
    }


# --- Use extracted data from cell 0 ---
df = df_extracted_1.copy()

# --- 1. PRE-PROCESSING / MERGING TEXT ---
print("Merging text columns into ProbableCause...")

def get_text(val):
    return str(val) if pd.notna(val) and str(val).strip() != "" else ""

new_probable_causes = []
for _, row in df.iterrows():
    base = get_text(row.get("ProbableCause", ""))

    def_event = get_text(row.get("Defining_Event_", ""))
    if def_event:
        base += f" | Defining Event: {def_event}"

    analysis = get_text(row.get("Analysis", ""))
    if analysis:
        base += f" | Analysis: {analysis}"

    pcf = get_text(row.get("Probable_Cause_and_Findings", ""))
    if pcf:
        base += f" | Probable Cause & Findings: {pcf}"

    new_probable_causes.append(base)

df["ProbableCause"] = new_probable_causes

# --- 2. COLUMN CLEANING & FILLING ---
print("Cleaning columns...")

if "EventDate" in df.columns:
    df["EventDate"] = df["EventDate"].apply(clean_date)

if "HighestInjuryLevel" in df.columns:
    df["HighestInjuryLevel"] = df["HighestInjuryLevel"].fillna("None")

num_cols = [
    "FatalInjuryCount",
    "SeriousInjuryCount",
    "MinorInjuryCount",
    "OnboardInjuryCount",
    "OnGroundInjuryCount",
]
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0.0)

if "EngineType" in df.columns:
    df["EngineType"] = df["EngineType"].fillna("unknown")

if "Scheduled" in df.columns:
    df["Scheduled"] = df["Scheduled"].replace("NSCH", "Non-scheduled").fillna("unknown")

if "FAR" in df.columns:
    df["FAR"] = df["FAR"].fillna("unknwon")

if "AirCraftDamage" in df.columns:
    df["AirCraftDamage"] = df["AirCraftDamage"].fillna("None")

if "WeatherCondition" in df.columns:
    df["WeatherCondition"] = df["WeatherCondition"].fillna("None")

if "Operator" in df.columns:
    df["Operator"] = df["Operator"].fillna("Unkown")

# --- 3. METEOROLOGICAL SPLIT ---
print("Splitting Meteorological Information...")
if "Meteorological_Information_and_Flight_Plan" in df.columns:
    meteo_data = df["Meteorological_Information_and_Flight_Plan"].apply(extract_meteo_info)
    meteo_df = pd.DataFrame(meteo_data.tolist())
    df = pd.concat([df, meteo_df], axis=1)
else:
    for c in [
        "Condition_of_Light",
        "Lowest_Cloud_Condition",
        "Wind_Info",
        "Lowest_Ceiling",
        "Visibility",
    ]:
        df[c] = "Unknown"

# --- 4. COLUMN REMOVAL ---
print("Dropping unwanted columns...")

cols_to_remove = [
    "ReportNo",
    "ReportType",
    "OriginalPublishedDate",
    "DocketOriginalPublishedDate",
    "EventID",
    "AirportName",
    "ReportStatus",
    "RepGenFlag",
    "MostRecentReportType",
    "DocketUrl",
    "ReportUrl",
    "Location_",
    "Accident_Number_",
    "Date_Time_",
    "Registration_",
    "Aircraft_",
    "Injuries_",
    "Flight_Conducted_Under_",
    "Aircraft_and_Owner_Operator_Information",
    "Meteorological_Information_and_Flight_Plan",
    "Wreckage_and_Impact_Information",
    "Aircraft_Damage_",
    "Administrative_Information",
    "Defining_Event_",
    "Analysis",
    "Probable_Cause_and_Findings",
    "Factual_Information",
    "Pilot_Information",
    "Airport_Information",
]

df.drop(columns=[c for c in cols_to_remove if c in df.columns], inplace=True, errors="ignore")

print(f"Saving to {FINAL_OUTPUT_FILE}...")
df.to_csv(FINAL_OUTPUT_FILE, index=False)
print("Done!")


Merging text columns into ProbableCause...
Cleaning columns...
Splitting Meteorological Information...
Dropping unwanted columns...
Saving to /Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/ntsb_with_extracted.csv...
Done!


In [4]:
import pandas as pd
import re

# Set options as requested
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

FILE_PATH = "/Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/ntsb_with_extracted.csv"


def main():
    print(f"Reading {FILE_PATH}...")
    df = pd.read_csv(FILE_PATH)

    # --- 1. Clean Condition_of_Light ---
    print("Cleaning Condition_of_Light...")
    df["Condition_of_Light"] = df["Condition_of_Light"].astype(str).str.replace(
        r"(\s|\\n|\n)*Observation Facility.*", "", regex=True
    ).str.strip()

    # --- 2. Clean Lowest_Cloud_Condition ---
    print("Cleaning Lowest_Cloud_Condition...")

    def clean_cloud(val):
        s = str(val).strip()
        if s.startswith("Wind Speed") or s.startswith("Visibility"):
            return "Unknown"
        return val

    df["Lowest_Cloud_Condition"] = df["Lowest_Cloud_Condition"].apply(clean_cloud)

    # --- 3. Clean Wind_Info ---
    print("Cleaning Wind_Info...")

    def clean_wind(val):
        s = str(val).strip()
        if s == "nan":
            return "Unknown"

        if s.startswith("Lowest Ceiling:"):
            return "Unknown"

        if "Turbulence Type" in s:
            s = s.split("Turbulence Type")[0].strip()
            s = s.rstrip(",")

        return s

    df["Wind_Info"] = df["Wind_Info"].apply(clean_wind)

    # --- 4. Clean Lowest_Ceiling ---
    print("Cleaning Lowest_Ceiling...")

    def clean_ceiling(val):
        if pd.isna(val) or str(val).lower() == "nan":
            return "Unknown"
        s = str(val).strip()
        if s.startswith("Visibility"):
            return "Unknown"
        return s

    df["Lowest_Ceiling"] = df["Lowest_Ceiling"].apply(clean_ceiling)

    # --- 5. Clean Visibility ---
    print("Cleaning Visibility...")

    def clean_visibility(val):
        s = str(val).strip()
        if s.startswith("Altimeter Setting"):
            return "Unknown"
        return val

    df["Visibility"] = df["Visibility"].apply(clean_visibility)

    print(f"Saving changes to {FILE_PATH}...")
    df.to_csv(FILE_PATH, index=False)
    print("Done!")


if __name__ == "__main__":
    main()


Reading /Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/ntsb_with_extracted.csv...
Cleaning Condition_of_Light...
Cleaning Lowest_Cloud_Condition...
Cleaning Wind_Info...
Cleaning Lowest_Ceiling...
Cleaning Visibility...
Saving changes to /Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/ntsb_with_extracted.csv...
Done!


In [5]:
print(len(df))

1890
